# Reviewer Revision — Stage B: All-Model External Transfer + Failure Diagnostics

This notebook addresses the reviewer's main scientific criticisms after the Stage-A reconciliation.

### Goals
1. Evaluate **all six trained model types** on Dreaddit, external Reddit, and Twitter:
   - Logistic Regression
   - Multinomial Naive Bayes
   - Linear SVM
   - Random Forest
   - DistilBERT
   - RoBERTa-base
2. Report a consistent metric set:
   - Accuracy
   - Macro-F1
   - Balanced accuracy
   - ROC-AUC
   - PR-AUC
   - Positive-class recall
   - Predicted-positive rate
3. Preserve model-selection integrity:
   - classical hyperparameters selected by 3-fold CV on **training data only**
   - transformer checkpoint selection by **Dreaddit validation Macro-F1 only**
4. Diagnose the Twitter failure:
   - probability / score distributions
   - predicted-positive rate
   - label-free EM prior-shift adjustment
   - source TF-IDF feature coverage and zero-vector rate
   - controlled Dreaddit length-truncation sensitivity
5. Save persistent models, predictions, figures, and result CSVs to Google Drive.

**Important:** This is the canonical reviewer-revision experiment. Do not update the manuscript until all cells complete.

In [ ]:
!pip -q install datasets transformers accelerate evaluate openpyxl scipy statsmodels joblib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json, math, hashlib, inspect, warnings, requests, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import load_dataset, Dataset
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer, set_seed
)

warnings.filterwarnings("ignore")

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. In Colab choose Runtime → Change runtime type → T4 GPU.")

print("GPU:", torch.cuda.get_device_name(0))

ROOT = Path("/content/drive/MyDrive/stress_detection_reviewer_revision/stageB")
MODELS_DIR = ROOT/"models"
PRED_DIR = ROOT/"predictions"
RESULTS_DIR = ROOT/"results"
FIG_DIR = ROOT/"figures"
for p in [ROOT, MODELS_DIR, PRED_DIR, RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

OLD_PHASE5 = Path("/content/drive/MyDrive/stress_detection_phase5_persistent")
OLD_RB_PRED = OLD_PHASE5/"predictions"

print("Persistent output:", ROOT)

## 1. Reconstruct the audited source and target datasets

In [ ]:
def normalize_text_for_audit(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"https?://\S+|www\.\S+", "<url>", text)
    text = re.sub(r"\s+", " ", text)
    return text

def text_hash(text):
    return hashlib.sha256(normalize_text_for_audit(text).encode("utf-8")).hexdigest()

ds = load_dataset("andreagasparini/dreaddit")
train_raw = ds["train"].to_pandas()[["text","label"]].copy()
test = ds["test"].to_pandas()[["text","label"]].copy()

train_raw["text_hash"] = train_raw["text"].map(text_hash)
test["text_hash"] = test["text"].map(text_hash)
overlap = set(train_raw["text_hash"]) & set(test["text_hash"])

clean = train_raw.drop_duplicates("text_hash", keep="first")
clean = clean[~clean["text_hash"].isin(overlap)].reset_index(drop=True)

RAW = Path("/content/stageB_raw")
RAW.mkdir(exist_ok=True)
FILES = {
    "Reddit_Title":"Reddit_Title.xlsx",
    "Reddit_Combi":"Reddit_Combi.xlsx",
    "Twitter_Full":"Twitter_Full.xlsx",
    "Twitter_NonAdvert":"Twitter_Non-Advert.xlsx",
}
BASE = "https://raw.githubusercontent.com/SenticNet/stress-detection/master"

def prep(df):
    df = df.copy()
    df["text"] = df["text"].astype("string").str.strip()
    df.loc[df["text"]=="","text"] = pd.NA
    df["label"] = pd.to_numeric(df["label"],errors="coerce")
    df = df.dropna(subset=["text","label"]).copy()
    df["label"] = df["label"].astype(int)
    df["text_hash"] = df["text"].map(text_hash)
    return df

external = {}
for name, fn in FILES.items():
    p = RAW/fn
    if not p.exists():
        r = requests.get(f"{BASE}/{fn}", timeout=60); r.raise_for_status(); p.write_bytes(r.content)
    raw = pd.read_excel(p)
    if name=="Reddit_Title":
        text, label = raw["title"], raw["label"]
    elif name=="Reddit_Combi":
        text = (raw["title"].fillna("").astype(str).str.strip()+" "+
                raw["body"].fillna("").astype(str).str.strip()).str.strip()
        label = raw["label"]
    elif name=="Twitter_Full":
        text, label = raw["text"], raw["labels"]
    else:
        text, label = raw["text"], raw["label"]
    d = prep(pd.DataFrame({"text":text,"label":label}))
    nuniq = d.groupby("text_hash")["label"].nunique()
    conflicts = set(nuniq[nuniq>1].index)
    d = d[~d["text_hash"].isin(conflicts)].drop_duplicates("text_hash",keep="first").reset_index(drop=True)
    external[name] = d

EVAL = {"Dreaddit_Test":test.reset_index(drop=True), **external}

print("Leakage-clean train:", len(clean))
for k,v in EVAL.items():
    print(k, len(v), dict(v.label.value_counts().sort_index()))

## 2. Train/reuse all four classical models using training-only CV

In [ ]:
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

classical_specs = {
    "Logistic_Regression": (
        LogisticRegression(max_iter=2000),
        {"tfidf__max_features":[10000,20000], "clf__C":[0.5,1.0,2.0]}
    ),
    "Multinomial_NB": (
        MultinomialNB(),
        {"tfidf__max_features":[10000,20000], "clf__alpha":[0.25,0.5,1.0]}
    ),
    "Linear_SVM": (
        LinearSVC(),
        {"tfidf__max_features":[10000,20000], "clf__C":[0.5,1.0,2.0]}
    ),
    "Random_Forest": (
        RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        {
            "tfidf__max_features":[10000,20000],
            "clf__max_depth":[None,30],
            "clf__class_weight":[None,"balanced"]
        }
    )
}

classical_models = {}
selection_rows = []

for name,(clf,grid) in classical_specs.items():
    f = MODELS_DIR/f"{name}.joblib"
    meta = MODELS_DIR/f"{name}_selection.json"

    if f.exists() and meta.exists():
        print("Loading", name)
        model = joblib.load(f)
        info = json.loads(meta.read_text())
    else:
        print("Fitting", name)
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,max_df=0.98)),
            ("clf", clf)
        ])
        gs = GridSearchCV(pipe, grid, scoring="f1_macro", cv=CV, n_jobs=-1, refit=True)
        gs.fit(clean["text"].astype(str), clean["label"].astype(int))
        model = gs.best_estimator_
        info = {"best_cv_macro_f1":float(gs.best_score_), "best_params":gs.best_params_}
        joblib.dump(model,f)
        meta.write_text(json.dumps(info,indent=2))

    classical_models[name] = model
    selection_rows.append({"model":name, **info})

selection = pd.DataFrame(selection_rows)
display(selection)
selection.to_csv(RESULTS_DIR/"classical_training_only_selection.csv",index=False)

In [ ]:
def get_classical_score(model, X):
    clf = model.named_steps["clf"]
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:,1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    raise ValueError("No scoring method")

def metrics_from_predictions(y,pred,score):
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        "accuracy":accuracy_score(y,pred),
        "macro_f1":f1_score(y,pred,average="macro",zero_division=0),
        "balanced_accuracy":balanced_accuracy_score(y,pred),
        "macro_precision":precision_score(y,pred,average="macro",zero_division=0),
        "macro_recall":recall_score(y,pred,average="macro",zero_division=0),
        "positive_recall":recall_score(y,pred,pos_label=1,zero_division=0),
        "predicted_positive_rate":float(np.mean(pred==1)),
        "roc_auc":roc_auc_score(y,score),
        "pr_auc":average_precision_score(y,score),
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)
    }

classical_metric_rows=[]
for mname,model in classical_models.items():
    for dname,df in EVAL.items():
        y = df.label.to_numpy()
        pred = model.predict(df.text.astype(str))
        score = get_classical_score(model, df.text.astype(str))
        row = {"model":mname,"seed":np.nan,"dataset":dname,
               **metrics_from_predictions(y,pred,score)}
        classical_metric_rows.append(row)

        pd.DataFrame({
            "text":df.text.astype(str),
            "true_label":y,
            "predicted_label":pred,
            "score":score
        }).to_csv(PRED_DIR/f"{mname}_{dname}.csv",index=False)

classical_metrics = pd.DataFrame(classical_metric_rows)
display(classical_metrics.round(6))
classical_metrics.to_csv(RESULTS_DIR/"classical_external_metrics.csv",index=False)

## 3. Transformer preparation

A fixed 90:10 stratified split of the leakage-clean source training data is reused for every seed.
RoBERTa attempts to reuse the exact old persistent checkpoint that reproduces the saved Phase-5 Dreaddit predictions. If no usable checkpoint exists, it is trained again persistently.
DistilBERT is trained/reused persistently with the same protocol.

In [ ]:
tr_idx, va_idx = train_test_split(
    np.arange(len(clean)),
    test_size=0.10,
    random_state=42,
    stratify=clean["label"]
)
tr_df = clean.iloc[tr_idx][["text","label"]].reset_index(drop=True)
va_df = clean.iloc[va_idx][["text","label"]].reset_index(drop=True)

print(len(tr_df), len(va_df), va_df.label.value_counts().to_dict())

MODEL_SPECS = {
    "DistilBERT":"distilbert-base-uncased",
    "RoBERTa":"roberta-base"
}
SEEDS = [42,52,62]
MAX_LEN = 256

In [ ]:
def compute_metrics_tr(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits,axis=-1)
    return {"f1_macro":f1_score(labels,pred,average="macro",zero_division=0)}

def tokenize_df(df, tokenizer):
    ds = Dataset.from_pandas(df[["text","label"]], preserve_index=False)
    return ds.map(lambda b: tokenizer(b["text"],truncation=True,max_length=MAX_LEN), batched=True)

def training_args(output_dir, seed):
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=3,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        weight_decay=0.01,
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=3,
        report_to="none",
        seed=seed,
        data_seed=42,
        fp16=True
    )
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        kwargs["eval_strategy"]="epoch"
    else:
        kwargs["evaluation_strategy"]="epoch"
    return TrainingArguments(**kwargs)

def make_trainer(model, args, train_ds, val_ds, tokenizer):
    kwargs = dict(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics_tr
    )
    sig = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in sig:
        kwargs["processing_class"]=tokenizer
    else:
        kwargs["tokenizer"]=tokenizer
    return Trainer(**kwargs)

def latest_checkpoint(folder):
    cps = list(folder.glob("checkpoint-*"))
    if not cps: return None
    return max(cps,key=lambda p:int(p.name.split("-")[-1]))

def old_roberta_candidate_checkpoints(seed):
    # Search recursively for old checkpoint folders whose path contains the seed.
    candidates=[]
    if OLD_PHASE5.exists():
        for p in OLD_PHASE5.rglob("checkpoint-*"):
            pathstr = str(p).lower()
            if f"seed_{seed}" in pathstr or f"seed{seed}" in pathstr or f"/{seed}/" in pathstr:
                candidates.append(p)
    return sorted(set(candidates))

### Train/reuse transformer models and save score-level predictions

In [ ]:
transformer_metric_rows=[]
artifact_rows=[]

for short, checkpoint in MODEL_SPECS.items():
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    train_ds = tokenize_df(tr_df,tokenizer)
    val_ds = tokenize_df(va_df,tokenizer)

    for seed in SEEDS:
        set_seed(seed)
        final_dir = MODELS_DIR/short/f"seed_{seed}"/"final_model"
        work_dir = MODELS_DIR/short/f"seed_{seed}"/"checkpoints"
        final_dir.mkdir(parents=True,exist_ok=True)
        work_dir.mkdir(parents=True,exist_ok=True)

        model_source = None

        # If a final canonical model exists, reuse it.
        config_file = final_dir/"config.json"
        if config_file.exists():
            print(f"Loading canonical {short} seed {seed}")
            model = AutoModelForSequenceClassification.from_pretrained(final_dir)
            model_source = "stageB_final_model"
        else:
            # For RoBERTa, first try old persistent checkpoints and pick one that reproduces old Dreaddit hard predictions.
            matched_old = False
            if short=="RoBERTa":
                old_pred_file = OLD_RB_PRED/f"RB_{seed}_Dreaddit_Test.csv"
                if old_pred_file.exists():
                    old_labels = pd.read_csv(old_pred_file)["predicted_label"].to_numpy()
                    te_ds_tmp = tokenize_df(test.reset_index(drop=True),tokenizer)
                    for cand in old_roberta_candidate_checkpoints(seed):
                        try:
                            cand_model = AutoModelForSequenceClassification.from_pretrained(cand)
                            args_tmp = training_args(work_dir,seed)
                            tr_tmp = make_trainer(cand_model,args_tmp,train_ds,val_ds,tokenizer)
                            logits = tr_tmp.predict(te_ds_tmp).predictions
                            pr = np.argmax(logits,axis=-1)
                            disagree = int(np.sum(pr!=old_labels))
                            print("Old checkpoint candidate", cand, "disagreements:", disagree)
                            del tr_tmp; torch.cuda.empty_cache()
                            if disagree==0:
                                model = cand_model
                                model.save_pretrained(final_dir)
                                tokenizer.save_pretrained(final_dir)
                                model_source = f"matched_old_checkpoint:{cand}"
                                matched_old=True
                                break
                            del cand_model; torch.cuda.empty_cache()
                        except Exception as e:
                            print("Skipping candidate",cand,"because",e)

            if not matched_old:
                print(f"Training/restarting {short} seed {seed}")
                model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)
                args = training_args(work_dir,seed)
                trainer = make_trainer(model,args,train_ds,val_ds,tokenizer)
                resume = latest_checkpoint(work_dir)
                trainer.train(resume_from_checkpoint=str(resume) if resume else None)
                trainer.save_model(final_dir)
                tokenizer.save_pretrained(final_dir)
                model = trainer.model
                model_source = "stageB_trained"
                del trainer; torch.cuda.empty_cache()

        # Create inference trainer
        args_inf = training_args(work_dir,seed)
        trainer = make_trainer(model,args_inf,train_ds,val_ds,tokenizer)

        # Validation prediction file for diagnostics
        val_out = trainer.predict(val_ds)
        val_logits = val_out.predictions
        np.save(RESULTS_DIR/f"{short}_seed{seed}_validation_logits.npy", val_logits)
        np.save(RESULTS_DIR/f"{short}_seed{seed}_validation_labels.npy", va_df.label.to_numpy())

        for dname,df in EVAL.items():
            dds = tokenize_df(df,tokenizer)
            out = trainer.predict(dds)
            logits = out.predictions
            probs = torch.softmax(torch.tensor(logits),dim=1).numpy()
            pred = np.argmax(logits,axis=1)
            ppos = probs[:,1]
            y = df.label.to_numpy()

            transformer_metric_rows.append({
                "model":short,"seed":seed,"dataset":dname,
                **metrics_from_predictions(y,pred,ppos)
            })

            pd.DataFrame({
                "text":df.text.astype(str),
                "true_label":y,
                "predicted_label":pred,
                "p_pos":ppos,
                "logit_0":logits[:,0],
                "logit_1":logits[:,1]
            }).to_csv(PRED_DIR/f"{short}_seed{seed}_{dname}.csv",index=False)

        artifact_rows.append({"model":short,"seed":seed,"model_source":model_source})
        del trainer, model
        torch.cuda.empty_cache()

artifact_log = pd.DataFrame(artifact_rows)
transformer_metrics = pd.DataFrame(transformer_metric_rows)
display(artifact_log)
display(transformer_metrics.round(6))
artifact_log.to_csv(RESULTS_DIR/"transformer_artifact_log.csv",index=False)
transformer_metrics.to_csv(RESULTS_DIR/"transformer_seedlevel_metrics.csv",index=False)

## 4. Unified six-model results

In [ ]:
all_seed = pd.concat([classical_metrics,transformer_metrics],ignore_index=True)

# Mean/SD by model and dataset; single-run classical SD is NaN.
metric_cols = ["accuracy","macro_f1","balanced_accuracy","positive_recall","roc_auc","pr_auc","predicted_positive_rate"]
summary_rows=[]
for (m,d),g in all_seed.groupby(["model","dataset"]):
    row={"model":m,"dataset":d,"runs":len(g)}
    for col in metric_cols:
        row[f"{col}_mean"]=g[col].mean()
        row[f"{col}_sd"]=g[col].std(ddof=1) if len(g)>1 else np.nan
    summary_rows.append(row)

all_summary=pd.DataFrame(summary_rows)
display(all_summary.round(6))
all_seed.to_csv(RESULTS_DIR/"ALL_SIX_MODELS_seedlevel.csv",index=False)
all_summary.to_csv(RESULTS_DIR/"ALL_SIX_MODELS_summary.csv",index=False)

## 5. Label-only reference baselines

In [ ]:
source_prior = clean.label.mean()
rng = np.random.default_rng(42)
base_rows=[]
for name,df in EVAL.items():
    y=df.label.to_numpy()
    majority=np.full(len(y),np.bincount(y).argmax())
    allpos=np.ones(len(y),dtype=int)
    vals=[]
    for _ in range(1000):
        pr=rng.binomial(1,source_prior,len(y))
        vals.append(f1_score(y,pr,average="macro",zero_division=0))
    base_rows.append({
        "dataset":name,
        "majority_macro_f1":f1_score(y,majority,average="macro",zero_division=0),
        "all_positive_macro_f1":f1_score(y,allpos,average="macro",zero_division=0),
        "source_prior_random_macro_f1_mean":np.mean(vals),
        "source_prior_random_macro_f1_sd":np.std(vals,ddof=1)
    })
baselines=pd.DataFrame(base_rows)
display(baselines.round(6))
baselines.to_csv(RESULTS_DIR/"label_only_baselines_stageB.csv",index=False)

## 6. Diagnostic A: RoBERTa score behavior

This quantifies whether Twitter failure reflects loss of ranking ability, decision-boundary shift, or both.

In [ ]:
diag_rows=[]
for seed in SEEDS:
    for dname,df in EVAL.items():
        p=pd.read_csv(PRED_DIR/f"RoBERTa_seed{seed}_{dname}.csv")
        for label in [0,1]:
            s=p.loc[p.true_label==label,"p_pos"]
            diag_rows.append({
                "seed":seed,"dataset":dname,"true_class":label,"n":len(s),
                "mean_p_pos":s.mean(),"median_p_pos":s.median(),
                "q10":s.quantile(.10),"q90":s.quantile(.90)
            })
score_diag=pd.DataFrame(diag_rows)
display(score_diag.round(6))
score_diag.to_csv(RESULTS_DIR/"roberta_score_distribution_summary.csv",index=False)

In [ ]:
# Journal-ready probability histograms for the two Twitter datasets.
for dname in ["Twitter_Full","Twitter_NonAdvert"]:
    fig,ax=plt.subplots(figsize=(7.2,4.5))
    # Pool probabilities across seeds only for descriptive visualization.
    neg=[]; pos=[]
    for seed in SEEDS:
        p=pd.read_csv(PRED_DIR/f"RoBERTa_seed{seed}_{dname}.csv")
        neg.extend(p.loc[p.true_label==0,"p_pos"].tolist())
        pos.extend(p.loc[p.true_label==1,"p_pos"].tolist())
    bins=np.linspace(0,1,31)
    ax.hist(neg,bins=bins,alpha=.55,density=True,label="Stress-negative")
    ax.hist(pos,bins=bins,alpha=.55,density=True,label="Stress-positive")
    ax.axvline(.5,linestyle="--",linewidth=1,label="Decision threshold 0.5")
    ax.set_xlabel("RoBERTa predicted probability of stress")
    ax.set_ylabel("Density")
    ax.set_title(dname.replace("_"," "))
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR/f"RoBERTa_probability_distribution_{dname}.png",dpi=300,bbox_inches="tight")
    plt.close(fig)
print("Saved Twitter probability-distribution figures.")

## 7. Diagnostic B: Label-free prior-shift adjustment (EM)

This uses only:
- source class prior,
- target **unlabeled** model probabilities.

Target labels are used only afterward to evaluate the adjusted predictions.
This is a sensitivity analysis under the label-shift assumption, not proof that label shift is the sole cause.

In [ ]:
def em_label_shift_binary(p1, source_prior1, max_iter=500, tol=1e-8, eps=1e-6):
    p1=np.clip(np.asarray(p1,float),eps,1-eps)
    Ps=np.c_[1-p1,p1]
    qs=np.array([1-source_prior1,source_prior1],float)
    qt=qs.copy()

    for _ in range(max_iter):
        w=Ps*(qt/qs)
        post=w/w.sum(axis=1,keepdims=True)
        new_q=post.mean(axis=0)
        new_q=np.clip(new_q,eps,1-eps)
        new_q=new_q/new_q.sum()
        if np.max(np.abs(new_q-qt))<tol:
            qt=new_q; break
        qt=new_q

    w=Ps*(qt/qs)
    adj=w/w.sum(axis=1,keepdims=True)
    return adj[:,1], qt[1]

prior_rows=[]
source_prior1=tr_df.label.mean()

for seed in SEEDS:
    for dname in ["Reddit_Title","Reddit_Combi","Twitter_Full","Twitter_NonAdvert"]:
        p=pd.read_csv(PRED_DIR/f"RoBERTa_seed{seed}_{dname}.csv")
        adj_p, estimated_prior = em_label_shift_binary(p.p_pos.values,source_prior1)
        adj_pred=(adj_p>=.5).astype(int)
        y=p.true_label.values
        prior_rows.append({
            "seed":seed,"dataset":dname,
            "source_prior_positive":source_prior1,
            "estimated_target_prior_positive_unlabeled":estimated_prior,
            "actual_target_prior_positive_for_evaluation_only":y.mean(),
            "original_macro_f1":f1_score(y,p.predicted_label,average="macro",zero_division=0),
            "adjusted_macro_f1":f1_score(y,adj_pred,average="macro",zero_division=0),
            "delta_macro_f1":f1_score(y,adj_pred,average="macro",zero_division=0)-
                             f1_score(y,p.predicted_label,average="macro",zero_division=0),
            "original_positive_recall":recall_score(y,p.predicted_label,pos_label=1,zero_division=0),
            "adjusted_positive_recall":recall_score(y,adj_pred,pos_label=1,zero_division=0),
            "adjusted_predicted_positive_rate":adj_pred.mean()
        })

prior_adj=pd.DataFrame(prior_rows)
display(prior_adj.round(6))
prior_adj.to_csv(RESULTS_DIR/"roberta_label_shift_EM_sensitivity.csv",index=False)

## 8. Diagnostic C: Source TF-IDF feature coverage

In [ ]:
# Use a fixed source TF-IDF vocabulary for a model-independent lexical shift diagnostic.
coverage_vec=TfidfVectorizer(
    lowercase=True,ngram_range=(1,2),min_df=2,max_df=.98,max_features=10000
)
coverage_vec.fit(clean.text.astype(str))
analyzer=coverage_vec.build_analyzer()
vocab=set(coverage_vec.vocabulary_.keys())

coverage_rows=[]
for name,df in EVAL.items():
    total=0; matched=0; zeros=0; nz_counts=[]
    X=coverage_vec.transform(df.text.astype(str))
    nz=np.diff(X.indptr)
    zeros=int(np.sum(nz==0))
    for txt in df.text.astype(str):
        feats=analyzer(txt)
        total+=len(feats)
        matched+=sum(f in vocab for f in feats)
    coverage_rows.append({
        "dataset":name,
        "source_tfidf_feature_coverage":matched/total if total else np.nan,
        "zero_vector_rate":zeros/len(df),
        "mean_nonzero_source_features":float(np.mean(nz)),
        "median_nonzero_source_features":float(np.median(nz))
    })
coverage=pd.DataFrame(coverage_rows)
display(coverage.round(6))
coverage.to_csv(RESULTS_DIR/"source_tfidf_feature_coverage.csv",index=False)

## 9. Diagnostic D: Controlled input-length sensitivity

The Dreaddit test set is truncated to the pooled median whitespace-token length of the two cleaned Twitter datasets.
The labels and original Dreaddit test cases stay unchanged. This is a sensitivity analysis for input length, not a simulation of Twitter language.

In [ ]:
twitter_lengths=np.concatenate([
    external["Twitter_Full"].text.astype(str).str.split().map(len).values,
    external["Twitter_NonAdvert"].text.astype(str).str.split().map(len).values
])
MED_TW=int(np.median(twitter_lengths))
print("Pooled Twitter median word length:",MED_TW)

trunc_test=test.reset_index(drop=True).copy()
trunc_test["text"]=trunc_test.text.astype(str).apply(lambda s:" ".join(s.split()[:MED_TW]))

length_rows=[]

# Classical
for mname,model in classical_models.items():
    for variant,df in [("original",test.reset_index(drop=True)),("twitter_length_truncated",trunc_test)]:
        y=df.label.values
        pred=model.predict(df.text.astype(str))
        score=get_classical_score(model,df.text.astype(str))
        length_rows.append({
            "model":mname,"seed":np.nan,"variant":variant,
            **metrics_from_predictions(y,pred,score)
        })

# Transformer
for short in ["DistilBERT","RoBERTa"]:
    for seed in SEEDS:
        model_dir=MODELS_DIR/short/f"seed_{seed}"/"final_model"
        tokenizer=AutoTokenizer.from_pretrained(model_dir)
        model=AutoModelForSequenceClassification.from_pretrained(model_dir)
        args=training_args(MODELS_DIR/short/f"seed_{seed}"/"length_tmp",seed)
        trainer=make_trainer(model,args,tokenize_df(tr_df,tokenizer),tokenize_df(va_df,tokenizer),tokenizer)

        for variant,df in [("original",test.reset_index(drop=True)),("twitter_length_truncated",trunc_test)]:
            dds=tokenize_df(df,tokenizer)
            out=trainer.predict(dds)
            logits=out.predictions
            ppos=torch.softmax(torch.tensor(logits),dim=1).numpy()[:,1]
            pred=np.argmax(logits,axis=1)
            y=df.label.values
            length_rows.append({
                "model":short,"seed":seed,"variant":variant,
                **metrics_from_predictions(y,pred,ppos)
            })
        del trainer,model; torch.cuda.empty_cache()

length_df=pd.DataFrame(length_rows)
display(length_df[["model","seed","variant","macro_f1","positive_recall","roc_auc"]].round(6))
length_df.to_csv(RESULTS_DIR/"length_control_dreaddit.csv",index=False)

## 10. Compact reviewer-revision summary

In [ ]:
# Focused summary for decision-making before rewriting.
rows=[]

for dname in EVAL:
    sub=all_summary[all_summary.dataset==dname].sort_values("macro_f1_mean",ascending=False)
    best=sub.iloc[0]
    rows.append({
        "dataset":dname,
        "best_model":best.model,
        "best_macro_f1":best.macro_f1_mean,
        "roberta_macro_f1":float(sub.loc[sub.model=="RoBERTa","macro_f1_mean"].iloc[0]),
        "distilbert_macro_f1":float(sub.loc[sub.model=="DistilBERT","macro_f1_mean"].iloc[0]),
        "best_classical_model":sub[sub.model.isin(["Logistic_Regression","Multinomial_NB","Linear_SVM","Random_Forest"])].iloc[0].model,
        "best_classical_macro_f1":sub[sub.model.isin(["Logistic_Regression","Multinomial_NB","Linear_SVM","Random_Forest"])].iloc[0].macro_f1_mean
    })

decision=pd.DataFrame(rows)
display(decision.round(6))
decision.to_csv(RESULTS_DIR/"STAGE_B_DECISION_SUMMARY.csv",index=False)

print("\nSaved everything to:",ROOT)
print("\nPlease send screenshots of:")
print("1) classical_training_only_selection")
print("2) ALL_SIX_MODELS_summary")
print("3) roberta_label_shift_EM_sensitivity")
print("4) source_tfidf_feature_coverage")
print("5) length_control_dreaddit")
print("6) STAGE_B_DECISION_SUMMARY")